# Bloco 01 -- Leitura e Exploracao de Dados

**Aula 08 | Trilha Databricks | Jornada de Dados**

---

Este e o primeiro notebook de PySpark da trilha. Ate agora voce usou Databricks principalmente via SQL, configurou Unity Catalog, criou pipelines com DLT e modelou dados em Medallion. Agora vamos mergulhar na **PySpark DataFrame API**.

Neste bloco voce vai aprender a:
- Carregar tabelas do catalogo com `spark.table()`
- Ler arquivos em diferentes formatos com `spark.read`
- Definir schemas explicitos com `StructType`
- Explorar DataFrames com `printSchema()`, `describe()`, `display()` e mais
- Comparar DataFrame API com `spark.sql()`

**Pre-requisito:** O script `dataset/script.sql` ja foi executado e as tabelas Northwind existem no catalogo.

## 1. Setup

In [0]:
# pyspark.sql.functions — modulo principal de funcoes do PySpark.
# Contem funcoes para manipulacao de colunas, agregacoes, strings, datas, etc.
# Importamos como "F" por convencao para evitar conflito com funcoes built-in do Python
# (ex: F.col(), F.sum(), F.max() em vez de col(), sum(), max()).
from pyspark.sql import functions as F

# pyspark.sql.types — modulo de tipos de dados do PySpark.
# Usado para definir schemas explicitos com StructType/StructField.
# Cada tipo corresponde a um tipo SQL:
#   StringType  → VARCHAR    | IntegerType → INT (32 bits)
#   ShortType   → SMALLINT   | FloatType   → FLOAT (32 bits)
#   DateType    → DATE       | DoubleType  → DOUBLE (64 bits)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType,
    DateType,
    ShortType,
)

# Altere para o nome do seu catalogo
CATALOG = "northwind"
SCHEMA = "bronze"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Criar Volume para armazenar arquivos (substitui DBFS /tmp)
spark.sql("CREATE VOLUME IF NOT EXISTS demo_files")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/demo_files"
print(f"Volume disponível em: {VOLUME_PATH}")

## 2. Carregando tabelas com `spark.table()`

A forma mais direta de carregar uma tabela que ja existe no Unity Catalog e usando `spark.table()`. Como voce ja executou o `script.sql`, todas as tabelas Northwind estao disponiveis.

`spark.table()` retorna um **DataFrame** -- a estrutura central do PySpark para manipulacao de dados.

In [0]:
# Carregando tabelas do catalogo
orders = spark.table("orders")
customers = spark.table("customers")

# Verificando o tipo -- e um DataFrame
print(type(orders))

In [0]:
# Visualizacao rapida com display() -- funcao nativa do Databricks
display(orders)

## 3. Lendo arquivos CSV com `spark.read.csv()`

Nem sempre os dados estao em tabelas do catalogo. Muitas vezes voce recebe arquivos CSV, JSON ou Parquet. O PySpark tem o `spark.read` para lidar com isso.

Para demonstrar, vamos **escrever** a tabela `orders` como CSV e depois **ler de volta**. Assim o exemplo fica auto-contido.

In [0]:
# Escrevendo orders como CSV para demonstracao
orders.write.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    mode="overwrite",
)

In [0]:
# Lendo o CSV com inferSchema=True
# O Spark tenta "adivinhar" os tipos das colunas
orders_csv = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    inferSchema=True,
)

orders_csv.printSchema()

> **Observacao:** Compare o schema inferido com o original. O `inferSchema` faz uma passada extra nos dados para detectar tipos, e nem sempre acerta -- por exemplo, pode inferir `integer` quando o tipo correto e `short`, ou `string` quando deveria ser `date`. Em producao, sempre defina o schema explicitamente.

In [0]:
# Comparando: schema original da tabela vs schema inferido do CSV
print("=== Schema original (spark.table) ===")
orders.printSchema()

print("\n=== Schema inferido (spark.read.csv) ===")
orders_csv.printSchema()

## 4. Schema explicito com `StructType`

Em producao, voce **nunca** deve confiar no `inferSchema`. A forma segura e definir o schema explicitamente usando `StructType` e `StructField`.

Cada `StructField` recebe:
- Nome da coluna (`string`)
- Tipo de dado (`StringType()`, `IntegerType()`, `FloatType()`, `DateType()`, etc.)
- Se aceita nulos (`True` / `False`)

Vamos definir o schema de `order_details` como exemplo.

In [0]:
# Primeiro, vamos escrever order_details como CSV
order_details = spark.table("order_details")

order_details.write.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    mode="overwrite",
)

In [0]:
# Definindo o schema explicito para order_details
order_details_schema = StructType([
    StructField("order_id", ShortType(), nullable=False),
    StructField("product_id", ShortType(), nullable=False),
    StructField("unit_price", FloatType(), nullable=False),
    StructField("quantity", ShortType(), nullable=False),
    StructField("discount", FloatType(), nullable=False),
])

In [0]:
# Lendo CSV com schema explicito -- sem inferencia
order_details_csv = spark.read.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    schema=order_details_schema,
)

order_details_csv.printSchema()

In [0]:
# Comparando: schema inferido vs schema explicito
order_details_inferido = spark.read.csv(
    f"{VOLUME_PATH}/order_details_csv",
    header=True,
    inferSchema=True,
)

print("=== Schema INFERIDO ===")
order_details_inferido.printSchema()

print("=== Schema EXPLICITO ===")
order_details_csv.printSchema()

> **Por que isso importa?** Veja as diferencas que o `inferSchema` introduziu:
>
> | Coluna | Inferido | Explicito | Diferenca |
> |---|---|---|---|
> | `order_id`, `product_id`, `quantity` | `integer` (32 bits) | `short` (16 bits) | 4 bytes vs 2 bytes por valor |
> | `unit_price`, `discount` | `double` (64 bits) | `float` (32 bits) | 8 bytes vs 4 bytes por valor |
>
> O `inferSchema` sempre "chuta pra cima" — usa `integer` em vez de `short` e `double` em vez de `float` porque, olhando so o CSV, ele nao tem como saber o range real dos dados.
>
> **Para o Northwind (830 linhas), nao faz diferenca perceptivel.** Mas em producao com bilhoes de linhas, cada byte conta:
> - **Memoria** — `short` ocupa metade do `integer`. Em 1 bilhao de linhas, sao ~2 GB a menos so nas colunas de ID
> - **Disco** — arquivos menores = leitura mais rapida do storage
> - **Shuffle** — menos dados trafegando entre executors em `join`/`groupBy`
>
> Alem do tamanho, tem a questao de **correcao semantica**: `float` tem ~7 digitos de precisao e `double` tem ~15. Para preco unitario, `float` basta. Para calculos financeiros de alta precisao, `double` ou `decimal` seria melhor.
>
> **Regra pratica:** em producao, sempre defina o schema explicitamente. Voce ganha tipo correto, menos memoria e leitura mais rapida (sem a passada extra do `inferSchema`).

## 5. `spark.table()` vs `spark.read` -- quando usar cada um

| Metodo | Quando usar | Exemplo |
|--------|-------------|----------|
| `spark.table("tabela")` | Tabela ja registrada no Unity Catalog | `spark.table("orders")` |
| `spark.read.format().load()` | Arquivos em volumes, DBFS ou storage externo | `spark.read.csv("/path/file.csv")` |

**Regra pratica:** Se a tabela esta no catalogo, use `spark.table()`. Se voce esta lendo arquivos brutos (CSV, JSON, Parquet), use `spark.read`.

In [0]:
# Abordagem 1: spark.table() -- tabela do catalogo
products = spark.table("products")
display(products.limit(5))

In [0]:
# Abordagem 2: spark.read -- arquivo CSV
# (usando o CSV que gravamos antes)
orders_from_file = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,
    inferSchema=True,
)
display(orders_from_file.limit(5))

## 6. Formatos de leitura

O PySpark suporta varios formatos de leitura. Vamos demonstrar os principais.

### 6.1 CSV

Formato texto, separado por virgulas. Simples, mas sem informacao de tipos -- precisa de `inferSchema` ou schema explicito.

In [0]:
# Ja demonstrado acima. Opcoes comuns:
df_csv = spark.read.csv(
    f"{VOLUME_PATH}/orders_csv",
    header=True,        # primeira linha e cabecalho
    inferSchema=True,   # inferir tipos (ou usar schema=)
    sep=",",            # separador (padrao e virgula)
    nullValue="NA",     # tratar "NA" como nulo
)
print(f"CSV: {df_csv.count()} linhas")

### 6.2 JSON

In [0]:
# Escrevendo orders como JSON
orders.write.json(
    f"{VOLUME_PATH}/orders_json",
    mode="overwrite",
)

# Lendo de volta
df_json = spark.read.json(f"{VOLUME_PATH}/orders_json")
print(f"JSON: {df_json.count()} linhas")
df_json.printSchema()

### 6.3 Parquet

Formato **colunar** e binario. Muito mais eficiente que CSV/JSON porque:
- Armazena tipos nativamente (sem necessidade de inferSchema)
- Compressao por coluna
- Leitura seletiva de colunas (column pruning)

In [0]:
# Escrevendo orders como Parquet
orders.write.parquet(
    f"{VOLUME_PATH}/orders_parquet",
    mode="overwrite",
)

# Lendo de volta -- note que nao precisa de inferSchema
df_parquet = spark.read.parquet(f"{VOLUME_PATH}/orders_parquet")
print(f"Parquet: {df_parquet.count()} linhas")
df_parquet.printSchema()

### 6.4 Delta

Formato nativo do Databricks. Parquet + log de transacoes. Suporte a ACID, time travel, schema evolution. E o formato que voce ja conhece das aulas anteriores.

In [0]:
# Escrevendo orders como Delta
orders.write.format("delta").mode("overwrite").save(
    f"{VOLUME_PATH}/orders_delta"
)

# Lendo de volta
df_delta = spark.read.format("delta").load(f"{VOLUME_PATH}/orders_delta")
print(f"Delta: {df_delta.count()} linhas")
df_delta.printSchema()

### Comparacao de formatos

| Formato | Tipos nativos | Compressao | Schema obrigatorio | Transacional | Uso recomendado |
|---------|:---:|:---:|:---:|:---:|---|
| CSV     | Nao | Nao | Nao | Nao | Importacao/exportacao simples |
| JSON    | Parcial | Nao | Nao | Nao | APIs, dados semi-estruturados |
| Parquet | Sim | Sim | Nao | Nao | Data lakes, consultas analiticas |
| Delta   | Sim | Sim | Nao | Sim | **Padrao no Databricks** -- sempre que possivel |

## 7. Exploracao de DataFrames

Agora que sabemos carregar dados, vamos explorar os DataFrames. Essas funcoes sao essenciais para entender a estrutura e o conteudo dos dados antes de qualquer transformacao.

### 7.1 `display(df)` -- Visualizacao nativa do Databricks

Funcao exclusiva do Databricks. Mostra os dados em formato tabular interativo com opcoes de grafico.

In [0]:
display(orders)

### 7.2 `df.printSchema()` -- Arvore de tipos

Mostra a estrutura do DataFrame em formato de arvore, com nome e tipo de cada coluna.

In [0]:
orders.printSchema()

In [0]:
customers.printSchema()

### 7.3 `df.describe()` -- Estatisticas descritivas

Retorna contagem, media, desvio padrao, minimo e maximo para colunas numericas.

In [0]:
display(orders.describe())

### 7.4 `df.dtypes` -- Lista de tuplas (nome, tipo)

In [0]:
# Retorna uma lista de tuplas (nome_coluna, tipo)
orders.dtypes

### 7.5 `df.columns` -- Lista de nomes das colunas

In [0]:
# Retorna uma lista Python com os nomes das colunas
print("Colunas de orders:", orders.columns)
print(f"Total: {len(orders.columns)} colunas")

print("\nColunas de customers:", customers.columns)
print(f"Total: {len(customers.columns)} colunas")

### 7.6 `df.count()` -- Contagem de linhas

In [0]:
# count() e uma ACTION -- dispara a execucao no cluster
print(f"orders: {orders.count()} registros")
print(f"customers: {customers.count()} registros")
print(f"order_details: {order_details.count()} registros")
print(f"products: {products.count()} registros")

### 7.7 `df.show()` -- Exibicao no console

Diferente do `display()`, o `show()` imprime no console como texto. Util para debugging e funciona fora do Databricks tambem.

Parametros:
- `n`: numero de linhas (padrao 20)
- `truncate`: truncar colunas longas (padrao True, 20 caracteres)

In [0]:
# show() com parametros
orders.show(5, truncate=False)

In [0]:
# show() com truncate padrao -- compare a diferenca
customers.show(5)

## 8. DataFrame API vs `spark.sql()` -- duas formas de fazer a mesma coisa

O PySpark oferece duas formas equivalentes de consultar dados:

1. **DataFrame API** -- encadeamento de metodos Python (`.filter()`, `.select()`, etc.)
2. **`spark.sql()`** -- escrever SQL puro como string

Ambas produzem o **mesmo plano de execucao**. A escolha e questao de estilo e contexto. Nesta aula, vamos focar na DataFrame API, mas sempre mostrando o equivalente SQL como referencia.

In [0]:
# Para usar spark.sql(), primeiro criamos uma temp view
products.createOrReplaceTempView("products_view")

In [0]:
# DataFrame API: produtos com preco > 50
resultado_df = (
    products
    .filter(F.col("unit_price") > 50)
    .select("product_name", "unit_price")
    .orderBy(F.col("unit_price").desc())
)

display(resultado_df)

In [0]:
# spark.sql(): mesma consulta em SQL puro
resultado_sql = spark.sql("""
    SELECT product_name, unit_price
    FROM products_view
    WHERE unit_price > 50
    ORDER BY unit_price DESC
""")

display(resultado_sql)

> **Resultado identico!** As duas abordagens geram o mesmo plano de execucao no Spark. A DataFrame API e mais "Pythonica" e permite composicao programatica. O `spark.sql()` e mais familiar para quem vem do SQL. Nesta trilha, vamos priorizar a DataFrame API -- que e o objetivo desta aula.

---

## Exercicios

Agora e sua vez! Tente resolver os exercicios abaixo antes de olhar as solucoes.

### Exercicio 1

Carregue a tabela `customers` e mostre:
1. O schema (`printSchema()`)
2. A contagem total de registros
3. Os 10 primeiros registros (`show()` com `truncate=False`)

In [0]:
# Exercicio 1 -- sua solucao aqui
customers_df = spark.table("customers")

print('== Schema ==')
customers_df.printSchema()

print('\nTotal de Registros: ', customers_df.count())

print('\n== Exibindo 10 Registros ==')
customers_df.show(10, truncate=False)


In [0]:
# Exercicio 1 -- Solucao
customers = spark.table("customers")

# 1. Schema
print("=== Schema ===")
customers.printSchema()

# 2. Contagem
print(f"\nTotal de clientes: {customers.count()}")

# 3. Primeiros 10 registros
print("\n=== 10 primeiros registros ===")
customers.show(10, truncate=False)

### Exercicio 2

Carregue a tabela `products`:
1. Escreva como CSV em `{VOLUME_PATH}/products_csv`
2. Leia o CSV com `inferSchema=True` e exiba o schema
3. Defina um schema explicito com `StructType` e leia novamente
4. Compare os dois schemas -- quais diferencas voce encontra?

**Dica:** As colunas de `products` sao: `product_id` (SMALLINT), `product_name` (STRING), `supplier_id` (SMALLINT), `category_id` (SMALLINT), `quantity_per_unit` (STRING), `unit_price` (FLOAT), `units_in_stock` (SMALLINT), `units_on_order` (SMALLINT), `reorder_level` (SMALLINT), `discontinued` (INT)

In [0]:
# Exercicio 2 -- sua solucao aqui
products_df = spark.table("products")

products_df.write.csv(f"{VOLUME_PATH}/products_csv", header=True, mode="overwrite")

products_schema = StructType(
    [
        StructField("product_id", ShortType(), False),
        StructField("product_name", StringType(), False),
        StructField("supplier_id", ShortType(), False),
        StructField("category_id", ShortType(), False),
        StructField("quantity_per_unit", StringType(), False),
        StructField("unit_price", FloatType(), False),
        StructField("units_in_stock", ShortType(), False),
        StructField("units_on_order", ShortType(), False),
        StructField("reorder_leve", ShortType(), False),
        StructField("discontinue", IntegerType(), False),
    ]
)

products_explicito = spark.read.csv(
    f"{VOLUME_PATH}/products_csv", header=True, schema=products_schema
)

products_inferido = spark.read.csv(
    f"{VOLUME_PATH}/products_csv", header=True, inferSchema=True
)

print(f"Schema INFERIDO\n {products_explicito.printSchema()}")
print(f"Schema Explícito\n {products_inferido.printSchema()}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ShortType,
    FloatType,
    IntegerType
)

# Ler tabela products
products_df = spark.table("products")

# Salvar como CSV
products_df.write.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    mode="overwrite"
)

# Criando schema manualmente
products_schema = StructType(
    [
        StructField("product_id", ShortType(), False),
        StructField("product_name", StringType(), False),
        StructField("supplier_id", ShortType(), False),
        StructField("category_id", ShortType(), False),
        StructField("quantity_per_unit", StringType(), False),
        StructField("unit_price", FloatType(), False),
        StructField("units_in_stock", ShortType(), False),
        StructField("units_on_order", ShortType(), False),
        StructField("reorder_level", ShortType(), False),
        StructField("discontinue", IntegerType(), False),
    ]
)


# Leitura usando schema explícito
products_explicito = spark.read.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    schema=products_schema
)


# Leitura usando inferência automática
products_inferido = spark.read.csv(
    f"{VOLUME_PATH}/products_csv",
    header=True,
    inferSchema=True
)


# Comparação dos schemas

print("Schema Explícito:")
products_explicito.printSchema()


print("\nSchema Inferido:")
products_inferido.printSchema()

### Exercicio 3

Escreva a mesma consulta de duas formas:

**Consulta:** Listar `first_name`, `last_name` e `hire_date` dos employees contratados depois de 1993-01-01, ordenados por data de contratacao.

1. Usando DataFrame API (`.filter()`, `.select()`, `.orderBy()`)
2. Usando `spark.sql()` (crie uma temp view primeiro)

In [0]:
# Exercicio 3 -- sua solucao aqui
employees_0 = spark.table("employees")

resultado = (
    employees_0.filter(F.col("hire_date") > "1993-01-01")
    .orderBy("hire_date")
    .select("first_name", "last_name", "hire_date")
)

print("Resultado API Spark: ")
resultado.show(10, truncate=False)

employees_0.createOrReplaceTempView("employees_vw")

resultado_sql = spark.sql("""
    SELECT first_name, last_name, hire_date
    FROM employees_vw
    WHERE hire_date > '1993-01-01'
    ORDER BY hire_date                          
""")

print("Resultado SQL: ")
resultado_sql.show(10, truncate=False)

In [0]:
# Exercicio 3 -- Solucao

employees = spark.table("employees")

# Abordagem 1: DataFrame API
print("=== DataFrame API ===")
resultado_api = (
    employees
    .filter(F.col("hire_date") > "1993-01-01")
    .select("first_name", "last_name", "hire_date")
    .orderBy("hire_date")
)
resultado_api.show(truncate=False)

# Abordagem 2: spark.sql()
print("=== spark.sql() ===")
employees.createOrReplaceTempView("employees_view")

resultado_sql = spark.sql("""
    SELECT first_name, last_name, hire_date
    FROM employees_view
    WHERE hire_date > '1993-01-01'
    ORDER BY hire_date
""")
resultado_sql.show(truncate=False)

---

## Resumo do Bloco 01

Neste bloco voce aprendeu:

| Conceito | Funcao/Metodo |
|----------|---------------|
| Carregar tabela do catalogo | `spark.table("tabela")` |
| Ler arquivo CSV | `spark.read.csv(path, header=True, inferSchema=True)` |
| Ler arquivo JSON | `spark.read.json(path)` |
| Ler arquivo Parquet | `spark.read.parquet(path)` |
| Ler arquivo Delta | `spark.read.format("delta").load(path)` |
| Schema explicito | `StructType([StructField("col", Tipo(), nullable)])` |
| Visualizar dados | `display(df)`, `df.show(n, truncate=False)` |
| Inspecionar schema | `df.printSchema()`, `df.dtypes` |
| Listar colunas | `df.columns` |
| Contar linhas | `df.count()` |
| Estatisticas | `df.describe()` |
| SQL direto | `spark.sql("SELECT ...")` |
| Criar temp view | `df.createOrReplaceTempView("nome")` |

No proximo bloco, vamos aprender **transformacoes**: `select()`, `filter()`, `withColumn()`, funcoes de string, data e tratamento de nulos.